In [ ]:
# No Colab: clone o repositório e entre na pasta (os .py precisam estar na pasta atual)
# !git clone https://github.com/SEU_USUARIO/SEU_REPO.git
# %cd SEU_REPO


In [ ]:
pip install datasets

In [ ]:
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import set_seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

dataset = load_dataset("peluz/lener_br", revision="refs/pr/5")
print(dataset)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

def alinhar_labels_com_tokens(labels_originais, word_ids):
    labels_alinhadas = []
    id_palavra_anterior = None

    for word_id in word_ids:
        if word_id is None:
            # token especial ([CLS], [SEP], padding)
            labels_alinhadas.append(-100)
        elif word_id != id_palavra_anterior:
            # primeiro subtoken desta palavra
            labels_alinhadas.append(labels_originais[word_id])
        else:
            # subtoken extra da mesma palavra
            labels_alinhadas.append(-100)
        id_palavra_anterior = word_id

    return labels_alinhadas

def tokenizar_e_alinhar(exemplos):
    tokenized = tokenizer(
        exemplos["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )

    todas_labels_alinhadas = []
    for i, labels_originais in enumerate(exemplos["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        labels_alinhadas = alinhar_labels_com_tokens(labels_originais, word_ids)
        todas_labels_alinhadas.append(labels_alinhadas)

    tokenized["labels"] = todas_labels_alinhadas
    return tokenized

dataset_tokenizado = dataset.map(tokenizar_e_alinhar, batched=True)

In [ ]:
input_ids = dataset_tokenizado["train"][0]["input_ids"]
tokens_reais = tokenizer.convert_ids_to_tokens(input_ids)
labels_reais = dataset_tokenizado["train"][0]["labels"]

for tok, lab in zip(tokens_reais, labels_reais):
    print(f"{tok:15} {lab}")

In [ ]:
from transformers import AutoModelForTokenClassification

tags = dataset["train"].features["ner_tags"].feature
label_names = tags.names

id2label = {i: nome for i, nome in enumerate(label_names)}
label2id = {nome: i for i, nome in enumerate(label_names)}

model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
!pip install setuptools_scm
!pip install seqeval --no-build-isolation

In [ ]:
from transformers import DataCollatorForTokenClassification
import numpy as np
import evaluate

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [label_names[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    resultados = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": resultados["overall_precision"],
        "recall": resultados["overall_recall"],
        "f1": resultados["overall_f1"],
        "accuracy": resultados["overall_accuracy"],
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bertimbau-lener-br",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenizado["train"],
    eval_dataset=dataset_tokenizado["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
predictions_output = trainer.predict(dataset_tokenizado["test"])

logits = predictions_output.predictions
labels = predictions_output.label_ids
predictions = np.argmax(logits, axis=-1)

true_labels = [
    [label_names[l] for l in label if l != -100]
    for label in labels
]
true_predictions = [
    [label_names[p] for p, l in zip(pred, label) if l != -100]
    for pred, label in zip(predictions, labels)
]

resultados = seqeval.compute(predictions=true_predictions, references=true_labels)

# resultados vem com um dict por entidade + as chaves 'overall_*'
for entidade, metricas in resultados.items():
    if entidade.startswith("overall"):
        continue
    print(f"{entidade:20} precision={metricas['precision']:.3f}  recall={metricas['recall']:.3f}  f1={metricas['f1']:.3f}  n={metricas['number']}")

print("\nOverall F1:", resultados["overall_f1"])

# aplicação dos paper 'Extensive Error Analysis and a Learning-Based Evaluation of Medical Entity Recognition Systems to Approximate User Experience'

In [ ]:
from entidades import bio_para_entidades, spans_se_sobrepoem, texto_da_entidade, classificar_mismatches

- **Mismatch Type-1 (Complete False Positive):** O modelo prevê uma entidade que não foi anotada.
- **Mismatch Type-2 (Complete False Negative):** Uma entidade anotada manualmente é totalmente ignorada pelo modelo.
- **Mismatch Type-3 (Wrong label, Right span):** O modelo acerta os limites exatos do texto, mas atribui a categoria errada.
- **Mismatch Type-4 (Wrong label, Overlapping span):** Erro tanto na categoria quanto nos limites, embora haja uma interseção entre eles.
- **Mismatch Type-5 (Right label, Overlapping span):** **Foco principal.** O modelo acerta a categoria, mas os limites do texto divergem da anotação original.

In [ ]:
from collections import Counter

todos_detalhes = []
for tokens_sent, labels_sent, preds_sent in zip(dataset["test"]["tokens"], true_labels, true_predictions):
    entidades_gold = bio_para_entidades(labels_sent)
    entidades_pred = bio_para_entidades(preds_sent)
    todos_detalhes.extend(classificar_mismatches(entidades_gold, entidades_pred, tokens_sent))

contagem_final = Counter([d["tipo"] for d in todos_detalhes])
print(contagem_final)

Aplicando o "Automatic Judge"

In [ ]:
pares_positivos_treino = []

for exemplo in dataset["train"]:
    tokens = exemplo["tokens"]
    tags_numericas = exemplo["ner_tags"]
    tags_strings = [label_names[tag] for tag in tags_numericas]

    entidades = bio_para_entidades(tags_strings)

    for ent in entidades:
        texto = texto_da_entidade(ent, tokens)
        label = ent["label"]
        pares_positivos_treino.append((texto, label))

In [ ]:
!pip install -q spacy
!python -m spacy download pt_core_news_lg

In [ ]:
import spacy

nlp = spacy.load("pt_core_news_lg")

In [ ]:
from juiz import candidatos_other_da_frase, classificar_span, avaliar_erro_com_juiz

In [ ]:
teste_other = candidatos_other_da_frase(dataset["train"][0], label_names, nlp)

print("Frase:")
print(teste_other["frase"])

print("\nEntidades gold:")
for entidade in teste_other["gold"]:
    print(entidade)

print("\nCandidatos a other:")
for candidato in teste_other["candidatos_other"]:
    print(repr(candidato))

In [ ]:
import random
candidatos_other_train = []

for exemplo in dataset["train"]:
    resultado = candidatos_other_da_frase(exemplo, label_names, nlp)
    candidatos_other_train.extend(resultado["candidatos_other"])

# Remove duplicatas, preservando a primeira forma encontrada.
candidatos_other_train = list(dict.fromkeys(candidatos_other_train))

In [ ]:
from collections import Counter
import random

contagem_positivos = Counter(
    label for _, label in pares_positivos_treino
)

limite_other = round(
    sum(contagem_positivos.values()) / len(contagem_positivos)
)

other_amostrados = random.sample(
    candidatos_other_train,
    k=min(limite_other, len(candidatos_other_train))
)

pares_treino = (
    pares_positivos_treino
    + [(texto, "other") for texto in other_amostrados]
)

random.shuffle(pares_treino)

print("Limite de OTHER:", limite_other)
print("Total final:", len(pares_treino))
print("\nDistribuição final:")
print(Counter(label for _, label in pares_treino))

In [ ]:
pares_positivos_val = []
candidatos_other_val = []

for exemplo in dataset["validation"]:
    resultado = candidatos_other_da_frase(exemplo, label_names, nlp)

    # As entidades gold continuam sendo exemplos positivos.
    pares_positivos_val.extend(resultado["gold"])

    # Os noun chunks fora delas são candidatos a other.
    candidatos_other_val.extend(resultado["candidatos_other"])

# Remove duplicatas dos candidatos negativos.
candidatos_other_val = list(dict.fromkeys(candidatos_other_val))

contagem_positivos_val = Counter(
    label for _, label in pares_positivos_val
)

limite_other_val = round(
    sum(contagem_positivos_val.values()) / len(contagem_positivos_val)
)

other_val_amostrados = random.sample(
    candidatos_other_val,
    k=min(limite_other_val, len(candidatos_other_val))
)

pares_val = (
    pares_positivos_val
    + [(texto, "other") for texto in other_val_amostrados]
)

random.shuffle(pares_val)

print("Distribuição da validação:")
print(Counter(label for _, label in pares_val))

In [ ]:
tags_unicas = sorted({label for _, label in pares_treino})

id2label_aux = {i: label for i, label in enumerate(tags_unicas)}
label2id_aux = {label: i for i, label in enumerate(tags_unicas)}

print(tags_unicas)
print("Número de classes:", len(tags_unicas))

In [ ]:
# CHECKPOINT_AUXILIAR = "adalbertojunior/distilbert-portuguese-cased"  # mais rápida, menos testada - mais parecida com a proposta do paper
CHECKPOINT_AUXILIAR = "neuralmind/bert-base-portuguese-cased"          # opção completa - mais pesada, já validada

In [ ]:
textos_train = [texto for texto, _ in pares_treino]
labels_train = [label2id_aux[label] for _, label in pares_treino]

textos_val = [texto for texto, _ in pares_val]
labels_val = [label2id_aux[label] for _, label in pares_val]

In [ ]:
from datasets import Dataset

dataset_aux_train = Dataset.from_dict({
    "text": textos_train,
    "label": labels_train,
})

dataset_aux_val = Dataset.from_dict({
    "text": textos_val,
    "label": labels_val,
})

In [ ]:
from transformers import AutoTokenizer
tokenizer_aux = AutoTokenizer.from_pretrained(CHECKPOINT_AUXILIAR)

def tokenizar_aux(exemplos):
    return tokenizer_aux(exemplos["text"], truncation=True)

dataset_aux_train = dataset_aux_train.map(tokenizar_aux, batched=True)
dataset_aux_val = dataset_aux_val.map(tokenizar_aux, batched=True)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics_aux(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")

    return {"accuracy": acc, "f1": f1}

In [ ]:
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

modelo_aux = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT_AUXILIAR,
    num_labels=len(tags_unicas),
    id2label=id2label_aux,
    label2id=label2id_aux,
)

data_collator_aux = DataCollatorWithPadding(tokenizer=tokenizer_aux)

training_args_aux = TrainingArguments(
    output_dir="./classificador-auxiliar",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_aux = Trainer(
    model=modelo_aux,
    args=training_args_aux,
    train_dataset=dataset_aux_train,
    eval_dataset=dataset_aux_val,
    data_collator=data_collator_aux,
    processing_class=tokenizer_aux,
    compute_metrics=compute_metrics_aux,
)

In [ ]:
trainer_aux.train()

In [ ]:
resultados_juiz = [avaliar_erro_com_juiz(erro, modelo_aux, tokenizer_aux, id2label_aux) for erro in todos_detalhes]
resultados_por_tipo = {
    tipo: [resultado for resultado in resultados_juiz if resultado["tipo"] == tipo]
    for tipo in ["type-1", "type-2", "type-3", "type-4", "type-5"]
}

for tipo, resultados_tipo in resultados_por_tipo.items():
    print(f"{tipo}: {dict(Counter(r['decisao'] for r in resultados_tipo))}")

O **learning-based F1** segue o paper: apenas Type-5 aceitos pelo judge deixam
de ser penalizados. Types 1 a 4 permanecem erros. O F1 relaxado credita todos
os Type-5; o estrito não credita nenhum.

In [ ]:
from metricas import f1_por_contagens


quantidades = Counter(erro["tipo"] for erro in todos_detalhes)
total_gold = sum(
    len(bio_para_entidades(labels_sent)) for labels_sent in true_labels
)
total_pred = sum(
    len(bio_para_entidades(preds_sent)) for preds_sent in true_predictions
)

erros_gold = sum(quantidades[tipo] for tipo in ["type-2", "type-3", "type-4", "type-5"])
erros_pred = sum(quantidades[tipo] for tipo in ["type-1", "type-3", "type-4", "type-5"])
tp_exato = total_gold - erros_gold

assert tp_exato + erros_gold == total_gold
assert tp_exato + erros_pred == total_pred

aceitos_type5 = sum(
    resultado["decisao"] == "aceito"
    for resultado in resultados_por_tipo["type-5"]
)
rejeitados_type5 = quantidades["type-5"] - aceitos_type5

metricas_estritas = f1_por_contagens(tp_exato, erros_pred, erros_gold)
metricas_relaxadas = f1_por_contagens(
    tp_exato + quantidades["type-5"],
    quantidades["type-1"] + quantidades["type-3"] + quantidades["type-4"],
    quantidades["type-2"] + quantidades["type-3"] + quantidades["type-4"],
)
metricas_learning = f1_por_contagens(
    tp_exato + aceitos_type5,
    quantidades["type-1"] + quantidades["type-3"] + quantidades["type-4"] + rejeitados_type5,
    quantidades["type-2"] + quantidades["type-3"] + quantidades["type-4"] + rejeitados_type5,
)

print(f"Type-5 aceitos pelo judge: {aceitos_type5}/{quantidades['type-5']}")
print(f"F1 estrito:        {metricas_estritas['f1']:.3f}")
print(f"F1 relaxado:       {metricas_relaxadas['f1']:.3f}")
print(f"F1 learning-based: {metricas_learning['f1']:.3f}")
print(f"F1 estrito (seqeval): {resultados['overall_f1']:.3f}")

In [ ]:
aceitos_type3 = sum(r["decisao"] == "aceito" for r in resultados_por_tipo["type-3"])
rejeitados_type3 = quantidades["type-3"] - aceitos_type3

aceitos_type4 = sum(r["decisao"] == "aceito" for r in resultados_por_tipo["type-4"])
rejeitados_type4 = quantidades["type-4"] - aceitos_type4

aceitos_345 = aceitos_type3 + aceitos_type4 + aceitos_type5
rejeitados_345 = rejeitados_type3 + rejeitados_type4 + rejeitados_type5

metricas_estendidas = f1_por_contagens(
    tp_exato + aceitos_345,
    quantidades["type-1"] + rejeitados_345,
    quantidades["type-2"] + rejeitados_345,
)

aceitos_type1 = sum(r["decisao"] == "aceito" for r in resultados_por_tipo["type-1"])

print(f"Type-3 aceitos pelo judge: {aceitos_type3}/{quantidades['type-3']}")
print(f"Type-4 aceitos pelo judge: {aceitos_type4}/{quantidades['type-4']}")
print(f"Type-5 aceitos pelo judge: {aceitos_type5}/{quantidades['type-5']}")
print(f"F1 estendido (3+4+5 perdoados quando aceitos): {metricas_estendidas['f1']:.3f}")
print()
print(f"[não entra no F1 pois não tem entidade gold correspondente] Type-1 aceitos pelo judge: {aceitos_type1}/{quantidades['type-1']}")

In [ ]:
def mais_curto(lista, limite=160):
    if not lista:
        return None
    curtos = [d for d in lista if len(d["frase"]) <= limite]
    pool = curtos if curtos else lista
    return min(pool, key=lambda d: len(d["frase"]))


print(f"1. DESEMPENHO DO NER (BERTimbau + LeNER-Br): F1 geral no teste: {resultados['overall_f1']:.3f}")
print()
print("2. DISTRIBUIÇÃO DE MISMATCHES NO TESTE:")
print(contagem_final)
print()
print("3. EXEMPLO REAL DE CADA TIPO DE MISMATCH")


for tipo in ["type-1", "type-2", "type-3", "type-4", "type-5"]:
    exemplo = mais_curto([d for d in todos_detalhes if d["tipo"] == tipo])
    print(f"\n--- {tipo.upper()} ---")
    if exemplo is None:
        print("(nenhum caso no conjunto de teste)")
        continue
    print(f"Frase: {exemplo['frase']}")
    print(f"Gold : {exemplo['gold_texto']!r}  ({exemplo['gold_label']})")
    print(f"Pred : {exemplo['pred_texto']!r}  ({exemplo['pred_label']})")

print()
print("4. AUTOMATIC JUDGE — resultados por tipo")

resultados_por_tipo = {
    tipo: [r for r in resultados_juiz if r["tipo"] == tipo]
    for tipo in ["type-1", "type-2", "type-3", "type-4", "type-5"]
}

for tipo, resultados_juiz_tipo in resultados_por_tipo.items():
    contagem = Counter([r["decisao"] for r in resultados_juiz_tipo])
    print(f"\n--- {tipo.upper()} — {dict(contagem)} ---")

    for decisao in ["aceito", "rejeitado"]:
        exemplo = mais_curto([r for r in resultados_juiz_tipo if r["decisao"] == decisao])
        if exemplo is None:
            print(f"\n[{decisao}] (nenhum caso)")
            continue
        print(f"\n[{decisao}]")
        print(f"Frase: {exemplo['frase']}")
        print(f"Gold : {exemplo['gold_texto']!r}  ({exemplo['gold_label']})")
        print(f"Pred : {exemplo['pred_texto']!r}  ({exemplo['pred_label']})")
        print(f"Juiz classificou o texto previsto como: {exemplo['label_juiz']}")



# Implementação do segundo paper: Automatic Keyphrase Extraction: A Survey of the State of the Art

## Métricas de ranking (Seção 4.1 do paper)

O paper aponta que P/R/F com exact match trata cada keyphrase como decisão
binária, sem considerar a posição no ranking. Isso é relevante pro nosso
caso porque o projeto tem foco em busca/navegação, não só em "acertou ou não".

Implementando três métricas complementares ao F1 que já temos:
- R-precision: precisão nos top-n do ranking, onde n = número de gold do documento
- MRR: posição do primeiro acerto no ranking
- Bpref: penaliza não-relevantes que aparecem antes dos relevantes (tolera gold incompleto)

In [ ]:
from metricas import (
    r_precision, reciprocal_rank, bpref, avaliar_corpus,
    confiancas_por_token, montar_ranking,
)

In [ ]:
confiancas_todas_sentencas = confiancas_por_token(logits, labels)

lista_ranked_candidatos = []
lista_gold = []

for tokens_sent, labels_sent, preds_sent, confiancas_sent in zip(
    dataset["test"]["tokens"], true_labels, true_predictions, confiancas_todas_sentencas):
    entidades_pred = bio_para_entidades(preds_sent)
    entidades_gold = bio_para_entidades(labels_sent)

    ranking = montar_ranking(entidades_pred, confiancas_sent, tokens_sent)

    gold_textos = {texto_da_entidade(ent, tokens_sent) for ent in entidades_gold}

    lista_ranked_candidatos.append(ranking)
    lista_gold.append(gold_textos)

resultado_final = avaliar_corpus(lista_ranked_candidatos, lista_gold)
print(resultado_final)